# eamon_xgboost_final

This notebook builds a rolling 10-trading-day rebalance portfolio:

1. Load the shared price data with `portfolio_toolkit`.
2. Build a Fama-French 5-factor covariance model at each rebalance date.
3. Estimate factor betas with exponentially weighted least squares so recent observations matter more.
4. Shrink factor covariance and residual covariance with Ledoit-Wolf where possible.
5. Train an XGBoost model to predict 10-day cross-sectional outperformance.
6. Slightly tilt rolling minimum-variance weights toward the XGBoost signal.
7. Validate, backtest, write artifacts, and log to MLflow.

The minimum-variance optimizer is the backbone. XGBoost only gets a small weight budget through `TILT_ALPHA`.


## 0. Bootstrap And Dependencies

This cell locates the repository root from either a local notebook session or Colab-style working directory, then installs the toolkit and the extra notebook-only packages.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path


def is_repo_root(path: Path) -> bool:
    return (path / 'pyproject.toml').exists() and (path / 'src' / 'portfolio_toolkit').exists()


def find_repo_root() -> Path:
    candidates = []
    if 'repo_root' in globals():
        candidates.append(Path(repo_root).expanduser())
    if os.environ.get('PWD'):
        candidates.append(Path(os.environ['PWD']).expanduser())
    candidates.append(Path.cwd())
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            continue
        for path in [resolved, *resolved.parents]:
            if is_repo_root(path):
                return path
    raise RuntimeError('Could not locate repository root. Set repo_root manually and rerun this cell.')


repo_root = find_repo_root()
os.chdir(repo_root)
src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print('repo_root =', repo_root)
print('python =', sys.executable)
print('cwd =', Path.cwd())

# The repo dev install provides the toolkit. The other packages are notebook-local dependencies.
subprocess.check_call([
    sys.executable,
    '-m',
    'pip',
    'install',
    '-q',
    '-e',
    str(repo_root) + '[dev]',
    'xgboost',
    'cvxpy',
    'pandas-datareader',
    'scikit-learn',
    'yfinance',
])

print('Install complete')

## 1. Imports And Configuration

All tunable parameters are kept in one place. `PREDICTION_HORIZON_DAYS` and `REBALANCE_EVERY_DAYS` are both set to 10 as requested.

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import cvxpy as cp
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import pandas_datareader.data as pdr
import xgboost as xgb
import yfinance as yf
import portfolio_toolkit.backtest as _bt_module
from sklearn.covariance import LedoitWolf
from sklearn.metrics import accuracy_score, log_loss

from portfolio_toolkit import (
    PortfolioWeights,
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_model_submission,
    log_portfolio,
    make_forward_return_target,
    slice_split,
    split_dates,
    start_run,
    validate_weights_frame,
    write_backtest_artifacts,
)

DATASET_NAME = 'shared_set_1'
MODEL_NAME = 'eamon_xgboost_final'
MINVAR_MODEL_NAME = 'eamon_xgboost_final_minvar'

PREDICTION_HORIZON_DAYS = 20
REBALANCE_EVERY_DAYS = 10
FF_WINDOW = 252
EW_HALFLIFE = 63
TILT_ALPHA = 0.25
MIN_WEIGHT = 0.0
MAX_WEIGHT = 0.15
MIN_FACTOR_OBS = 126
RANDOM_STATE = 42

OUTPUT_DIR = repo_root / 'runs' / 'eamon_xgboost_final'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
IV_CACHE_PATH = OUTPUT_DIR / 'current_iv_snapshot.csv'
MODEL_ARTIFACT_PATH = OUTPUT_DIR / f'{MODEL_NAME}.json'

FACTOR_COLS = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

TILT_FEATURE_NAMES = [
    'cs_rank_vol_60d',
    'downside_vol_20d',
    'cs_rank_beta_60d_spy',
    'spy_return_60d',
    'vol_10d',
    'return_dispersion_20d',
    'spy_vol_60d',
    'vol_5d',
    'spy_vol_20d',
    'price_to_sma_200d',
    'cs_rank_distance_to_60d_high',
    'cs_rank_momentum_120d',
    'vol_of_vol_60d',
    'price_to_sma_50d',
    'current_iv',
    'price_to_sma_20d',
    'vol_20d',
    'atr_14',
    'volume_zscore_20d',
    'mom_60_20_divergence',
    'dollar_volume_ratio_20d',
    'vol_of_vol_20d',
    'upside_vol_20d',
    'price_to_sma_10d',
    'vol_of_vol_10d',
    'current_iv_missing',
    'volume_zscore_60d',
    'volume_change_5d',
    'volume_change_1d',
    'volume_change_20d',
]

TOOLKIT_FEATURE_NAMES = [
    'downside_vol_20d',
    'vol_10d',
    'vol_5d',
    'price_to_sma_200d',
    'price_to_sma_50d',
    'price_to_sma_20d',
    'vol_20d',
    'atr_14',
    'volume_zscore_20d',
    'dollar_volume_ratio_20d',
    'upside_vol_20d',
    'price_to_sma_10d',
    'volume_zscore_60d',
    'volume_change_5d',
    'volume_change_1d',
    'volume_change_20d',
    'vol_60d',
    'beta_60d_spy',
    'distance_to_60d_high',
    'momentum_120d',
    'momentum_60d',
    'momentum_20d',
]

if len(TILT_FEATURE_NAMES) != len(set(TILT_FEATURE_NAMES)):
    raise ValueError('Duplicate feature names found in TILT_FEATURE_NAMES')

assert PREDICTION_HORIZON_DAYS == 20
assert REBALANCE_EVERY_DAYS == 10

print('Dataset:', DATASET_NAME)
print('Prediction horizon:', PREDICTION_HORIZON_DAYS)
print('Rebalance cadence:', REBALANCE_EVERY_DAYS)
print('FF window:', FF_WINDOW)
print('EW halflife:', EW_HALFLIFE)
print('Tilt alpha:', TILT_ALPHA)
print('Feature count:', len(TILT_FEATURE_NAMES))


## 1a. Notebook-Local Backtest Alignment Patch

This cell keeps the backtest row-normalization fix inside the notebook.

In [ ]:
import portfolio_toolkit.backtest as _bt_module
import pandas as pd


def _fixed_align_weights_to_prices(weights, price_index):
    aligned_rows = []
    aligned_index = []
    for date_value, row in weights.sort_index().iterrows():
        requested = pd.Timestamp(date_value)
        position = price_index.searchsorted(requested, side="left")
        if position >= len(price_index):
            continue
        aligned_rows.append(row)
        aligned_index.append(pd.Timestamp(price_index[position]))
    if not aligned_rows:
        raise ValueError("no weight rows align to the available trading calendar")
    aligned = pd.DataFrame(aligned_rows, index=pd.DatetimeIndex(aligned_index))
    aligned.index.name = "date"
    aligned = aligned.groupby(level=0).last()
    aligned = aligned.div(aligned.sum(axis=1), axis=0)
    return _bt_module.validate_weights_frame(aligned)


_bt_module._align_weights_to_prices = _fixed_align_weights_to_prices
print("Patch applied")

## 2. Load Shared Prices And Splits

The shared dataset and split helpers keep the run comparable with the rest of the repository. The benchmark ticker is excluded from the investable universe.

In [ ]:
spec = get_dataset_spec(DATASET_NAME, repo_root=repo_root)
splits = split_dates(DATASET_NAME, repo_root=repo_root)
prices = load_prices(DATASET_NAME, repo_root=repo_root)

train_start, train_end = splits['train']
val_start, val_end = splits['val']
test_start, test_end = splits['test']

adj_close = prices.pivot(index='date', columns='ticker', values='adj_close').sort_index()
adj_close.index = pd.to_datetime(adj_close.index)
returns_wide = adj_close.pct_change().dropna(how='all')

portfolio_tickers = [ticker for ticker in spec.tickers if ticker in adj_close.columns]
benchmark_ticker = spec.benchmark_ticker

print('Dataset display name:', spec.name)
print('Benchmark:', benchmark_ticker)
print('Configured tickers:', len(spec.tickers))
print('Tickers with price columns:', len(portfolio_tickers))
print('Date range:', prices['date'].min(), '->', prices['date'].max())
print('Splits:', splits)
display(prices.head())

## 3. Current Implied Volatility Feature

Warning: the repository only stores historical OHLCV data. This notebook uses current yfinance option-chain implied volatility as requested. That value is not historical point-in-time IV, so it is not a clean historical backtest signal. It is cached in the run folder and repeated as a static ticker-level feature across all dates.

In [ ]:
def _nearest_atm_iv(option_frame: pd.DataFrame, spot: float) -> float:
    if option_frame.empty or not np.isfinite(spot):
        return np.nan
    frame = option_frame.loc[:, ['strike', 'impliedVolatility']].copy()
    frame['impliedVolatility'] = pd.to_numeric(frame['impliedVolatility'], errors='coerce')
    frame = frame.dropna(subset=['strike', 'impliedVolatility'])
    if frame.empty:
        return np.nan
    idx = (frame['strike'] - spot).abs().idxmin()
    return float(frame.loc[idx, 'impliedVolatility'])


def fetch_current_iv_snapshot(tickers, latest_prices, cache_path: Path, refresh: bool = False) -> pd.DataFrame:
    if cache_path.exists() and not refresh:
        cached = pd.read_csv(cache_path)
        cached['ticker'] = cached['ticker'].astype(str).str.upper()
        if 'current_iv' not in cached.columns:
            median_iv = cached['current_iv_raw'].median(skipna=True) if 'current_iv_raw' in cached.columns else np.nan
            if not np.isfinite(median_iv):
                median_iv = 0.25
            cached['current_iv_missing'] = cached.get('current_iv_raw', pd.Series(np.nan, index=cached.index)).isna().astype(float)
            cached['current_iv'] = cached.get('current_iv_raw', pd.Series(np.nan, index=cached.index)).fillna(median_iv)
        return cached

    rows = []
    for idx, ticker in enumerate(tickers, start=1):
        spot = float(latest_prices.get(ticker, np.nan))
        current_iv = np.nan
        expiration = None
        try:
            yf_ticker = yf.Ticker(ticker)
            expirations = list(yf_ticker.options)
            if expirations:
                expiration = expirations[0]
                chain = yf_ticker.option_chain(expiration)
                call_iv = _nearest_atm_iv(chain.calls, spot)
                put_iv = _nearest_atm_iv(chain.puts, spot)
                current_iv = float(np.nanmean([call_iv, put_iv]))
        except Exception as exc:
            print(f'IV fetch failed for {ticker}: {exc}')

        rows.append(
            {
                'ticker': ticker,
                'current_iv_raw': current_iv,
                'iv_expiration': expiration,
                'iv_spot': spot,
            }
        )
        if idx % 25 == 0 or idx == len(tickers):
            print(f'Fetched IV for {idx}/{len(tickers)} tickers')

    iv = pd.DataFrame(rows)
    median_iv = iv['current_iv_raw'].median(skipna=True)
    if not np.isfinite(median_iv):
        median_iv = 0.25
    iv['current_iv_missing'] = iv['current_iv_raw'].isna().astype(float)
    iv['current_iv'] = iv['current_iv_raw'].fillna(median_iv)
    iv.to_csv(cache_path, index=False)
    return iv


latest_prices = adj_close.loc[:, portfolio_tickers].ffill().iloc[-1]
iv_snapshot = fetch_current_iv_snapshot(portfolio_tickers, latest_prices, IV_CACHE_PATH, refresh=False)

print('IV rows:', len(iv_snapshot))
print('Missing IV count:', int(iv_snapshot['current_iv_missing'].sum()))
display(iv_snapshot.head())

## 4. Build XGBoost Features And 10-Day Target

The feature set is limited to the requested families: volatility, volatility of volatility, current implied volatility, volume, and simple moving-average distance. The target is the 10-day forward return quintile within each date.

In [ ]:
def add_vol_of_vol_features(feature_frame: pd.DataFrame, prices_frame: pd.DataFrame) -> pd.DataFrame:
    panel = prices_frame.sort_values(['ticker', 'date']).loc[:, ['date', 'ticker', 'adj_close']].copy()
    panel['return_1d_for_vov'] = panel.groupby('ticker')['adj_close'].pct_change()
    short_vol = panel.groupby('ticker')['return_1d_for_vov'].transform(
        lambda s: s.rolling(5, min_periods=5).std(ddof=0)
    )
    vov = panel.loc[:, ['date', 'ticker']].copy()
    for window in [10, 20, 60]:
        vov[f'vol_of_vol_{window}d'] = short_vol.groupby(panel['ticker']).transform(
            lambda s, window=window: s.rolling(window, min_periods=window).std(ddof=0)
        )
    return feature_frame.merge(vov, on=['date', 'ticker'], how='left')


def build_market_context(prices_frame: pd.DataFrame) -> pd.DataFrame:
    panel = prices_frame.sort_values(['ticker', 'date']).copy()
    panel['return_1d_local'] = panel.groupby('ticker', sort=False)['adj_close'].pct_change()
    wide_returns = panel.pivot(index='date', columns='ticker', values='return_1d_local').sort_index()

    spy = (
        panel.loc[panel['ticker'] == 'SPY', ['date', 'adj_close']]
        .drop_duplicates('date')
        .set_index('date')['adj_close']
        .sort_index()
    )
    market = pd.DataFrame(index=spy.index)
    market['spy_return_60d'] = spy.pct_change(60)
    spy_daily = spy.pct_change()
    market['spy_vol_20d'] = spy_daily.rolling(20, min_periods=20).std(ddof=0)
    market['spy_vol_60d'] = spy_daily.rolling(60, min_periods=60).std(ddof=0)

    tradable_cols = [col for col in wide_returns.columns if col != 'SPY']
    market['return_dispersion_20d'] = wide_returns[tradable_cols].std(axis=1, skipna=True).rolling(20, min_periods=20).mean()

    return market.reset_index().rename(columns={'index': 'date'})


def add_custom_features(features: pd.DataFrame) -> pd.DataFrame:
    features = features.copy()
    features['mom_60_20_divergence'] = features['momentum_60d'] - features['momentum_20d']
    for name in ['vol_60d', 'beta_60d_spy', 'distance_to_60d_high', 'momentum_120d']:
        features[f'cs_rank_{name}'] = features.groupby('date')[name].rank(pct=True, method='average')
    return features


feature_frame = build_features(prices, feature_names=TOOLKIT_FEATURE_NAMES)
feature_frame = add_vol_of_vol_features(feature_frame, prices)
feature_frame = feature_frame.merge(build_market_context(prices), on='date', how='left')
feature_frame = add_custom_features(feature_frame)
feature_frame = feature_frame.merge(
    iv_snapshot.loc[:, ['ticker', 'current_iv', 'current_iv_missing']],
    on='ticker',
    how='left',
)

# If an IV row was not returned for a ticker, use the same median-imputation policy.
median_iv = feature_frame['current_iv'].median(skipna=True)
feature_frame['current_iv_missing'] = feature_frame['current_iv_missing'].fillna(1.0)
feature_frame['current_iv'] = feature_frame['current_iv'].fillna(median_iv)

missing_tilt_features = [feature for feature in TILT_FEATURE_NAMES if feature not in feature_frame.columns]
if missing_tilt_features:
    raise KeyError(f'TILT_FEATURE_NAMES missing from feature_frame: {missing_tilt_features}')

# Build a scoring feature table independent of target availability. This avoids losing late-test
# rebalance snapshots where the future 10-day target is naturally unavailable.
model_features = (
    feature_frame
    .loc[feature_frame['ticker'].isin(portfolio_tickers), ['date', 'ticker'] + TILT_FEATURE_NAMES]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=TILT_FEATURE_NAMES)
    .sort_values(['date', 'ticker'])
    .reset_index(drop=True)
)

if len(TILT_FEATURE_NAMES) != len(set(TILT_FEATURE_NAMES)):
    raise ValueError('Duplicate feature names found in TILT_FEATURE_NAMES')
missing_model_features = [feature for feature in TILT_FEATURE_NAMES if feature not in model_features.columns]
if missing_model_features:
    raise KeyError(f'TILT_FEATURE_NAMES missing from model_features: {missing_model_features}')

# Rank-normalize continuous signals cross-sectionally by date. Keep the missing-IV flag binary.
rank_normalized_features = [feature for feature in TILT_FEATURE_NAMES if feature != 'current_iv_missing']
for feature in rank_normalized_features:
    model_features[feature] = model_features.groupby('date')[feature].transform(lambda x: x.rank(pct=True))

return_target = make_forward_return_target(prices, horizon=PREDICTION_HORIZON_DAYS)
target_col = f'forward_return_{PREDICTION_HORIZON_DAYS}d'

panel = (
    model_features
    .merge(return_target.loc[:, ['date', 'ticker', target_col]], on=['date', 'ticker'], how='inner')
    .dropna(subset=[target_col])
    .sort_values(['date', 'ticker'])
    .reset_index(drop=True)
)

panel['quintile'] = (
    panel.groupby('date')[target_col]
    .transform(lambda x: pd.qcut(x.rank(method='first'), 5, labels=False, duplicates='drop'))
)
panel = panel.dropna(subset=['quintile']).copy()
panel['quintile'] = panel['quintile'].astype(int)

train_panel = slice_split(panel, DATASET_NAME, 'train', repo_root=repo_root)
val_panel = slice_split(panel, DATASET_NAME, 'val', repo_root=repo_root)
test_feature_panel = slice_split(model_features, DATASET_NAME, 'test', repo_root=repo_root)

print('Model feature rows:', len(model_features))
print('Train rows:', len(train_panel))
print('Val rows:', len(val_panel))
print('Test feature rows:', len(test_feature_panel))
display(model_features.head())


## 5. Train XGBoost Quintile Classifier

The model is trained only on the training split. Validation is used for early stopping and diagnostics. The test target is not used in training or scoring.

In [ ]:
X_train = train_panel[TILT_FEATURE_NAMES]
y_train = train_panel['quintile']
X_val = val_panel[TILT_FEATURE_NAMES]
y_val = val_panel['quintile']

xgb_model = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    n_estimators=600,
    max_depth=4,
    learning_rate=0.02,
    subsample=0.75,
    colsample_bytree=0.75,
    min_child_weight=20,
    reg_lambda=3.0,
    reg_alpha=0.5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=0,
    eval_metric='mlogloss',
    early_stopping_rounds=40,
)

xgb_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
val_pred = xgb_model.predict(X_val)
val_proba = xgb_model.predict_proba(X_val)
val_accuracy = accuracy_score(y_val, val_pred)
val_log_loss = log_loss(y_val, val_proba, labels=[0, 1, 2, 3, 4])

print('Validation accuracy:', round(val_accuracy, 4))
print('Validation log loss:', round(val_log_loss, 4))
print('Naive five-class accuracy:', 0.2)
print('Best iteration:', getattr(xgb_model, 'best_iteration', None))

importance = pd.Series(xgb_model.feature_importances_, index=TILT_FEATURE_NAMES).sort_values(ascending=False)
display(importance.to_frame('importance'))

fig, ax = plt.subplots(figsize=(10, 6))
importance.sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('XGBoost Feature Importance')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 6. Fama-French 5-Factor Data

Daily FF5 factors are aligned to stock returns. Each rebalance uses only rows strictly before that rebalance date.

In [ ]:
ff_start = prices['date'].min().date().isoformat()
ff_end = prices['date'].max().date().isoformat()
print('Downloading FF5 daily factors:', ff_start, '->', ff_end)

ff5_raw = pdr.DataReader('F-F_Research_Data_5_Factors_2x3_daily', 'famafrench', start=ff_start, end=ff_end)[0]
ff5 = ff5_raw / 100.0
ff5.index = pd.to_datetime(ff5.index)
factors = ff5[FACTOR_COLS].copy()
rf = ff5['RF'].copy()

common_factor_dates = returns_wide.index.intersection(factors.index)
print('FF5 rows:', len(factors))
print('Common return/factor dates:', len(common_factor_dates))
display(ff5.tail())

## 7. Risk Model And Optimization Helpers

These helpers keep the rolling loop readable. Betas use exponentially weighted least squares. Factor and residual covariance use Ledoit-Wolf shrinkage when there are enough observations. Intercepts are shrunk toward their cross-sectional mean and stored for diagnostics.

In [ ]:
def exponential_sample_weights(n: int, halflife: float) -> np.ndarray:
    ages = np.arange(n - 1, -1, -1, dtype=float)
    weights = 0.5 ** (ages / float(halflife))
    return weights / weights.mean()


def weighted_factor_regression(y: np.ndarray, X: np.ndarray, weights: np.ndarray):
    X_design = np.column_stack([np.ones(len(X)), X])
    sqrt_w = np.sqrt(weights)
    beta, *_ = np.linalg.lstsq(X_design * sqrt_w[:, None], y * sqrt_w, rcond=None)
    fitted = X_design @ beta
    residuals = y - fitted
    return float(beta[0]), beta[1:].astype(float), residuals.astype(float)


def nearest_psd(matrix: np.ndarray, jitter: float = 1e-8) -> np.ndarray:
    sym = (matrix + matrix.T) / 2.0
    min_eig = float(np.linalg.eigvalsh(sym).min())
    if min_eig < jitter:
        sym = sym + np.eye(sym.shape[0]) * (abs(min_eig) + jitter)
    return sym


def estimate_ew_ff5_covariance(rebalance_date: pd.Timestamp):
    eligible_dates = common_factor_dates[common_factor_dates < pd.Timestamp(rebalance_date)]
    window_dates = eligible_dates[-FF_WINDOW:]
    if len(window_dates) < MIN_FACTOR_OBS:
        raise ValueError(f'Not enough factor observations before {rebalance_date}: {len(window_dates)}')

    window_returns = returns_wide.loc[window_dates, portfolio_tickers]
    window_factors = factors.loc[window_dates, FACTOR_COLS]
    window_rf = rf.loc[window_dates]
    X_all = window_factors.to_numpy(dtype=float)

    beta_rows = {}
    intercepts = {}
    residual_series = {}
    residual_vars = {}

    for ticker in portfolio_tickers:
        y_raw = window_returns[ticker] - window_rf
        valid = y_raw.notna() & window_factors.notna().all(axis=1) & window_rf.notna()
        if int(valid.sum()) < MIN_FACTOR_OBS:
            continue
        y = y_raw.loc[valid].to_numpy(dtype=float)
        X = window_factors.loc[valid, FACTOR_COLS].to_numpy(dtype=float)
        sample_weights = exponential_sample_weights(len(y), EW_HALFLIFE)
        intercept, beta, residuals = weighted_factor_regression(y, X, sample_weights)
        beta_rows[ticker] = beta
        intercepts[ticker] = intercept
        residual_series[ticker] = pd.Series(residuals, index=window_factors.loc[valid].index)
        residual_vars[ticker] = float(np.average(residuals ** 2, weights=sample_weights))

    fitted_tickers = list(beta_rows.keys())
    if len(fitted_tickers) < max(8, int(1.0 / MAX_WEIGHT) + 1):
        raise ValueError(f'Only {len(fitted_tickers)} tickers fitted before {rebalance_date}')

    B = np.vstack([beta_rows[ticker] for ticker in fitted_tickers])
    factor_lw = LedoitWolf().fit(window_factors.loc[:, FACTOR_COLS].dropna().to_numpy(dtype=float))
    F = factor_lw.covariance_

    residual_frame = pd.DataFrame(residual_series).loc[:, fitted_tickers]
    residual_centered = residual_frame - residual_frame.mean(skipna=True)
    residual_for_lw = residual_centered.fillna(0.0)

    residual_shrinkage = 1.0
    if residual_for_lw.shape[0] >= 20 and residual_for_lw.shape[1] >= 2:
        residual_lw = LedoitWolf().fit(residual_for_lw.to_numpy(dtype=float))
        E = residual_lw.covariance_
        residual_shrinkage = float(residual_lw.shrinkage_)
    else:
        E = np.diag([residual_vars[ticker] for ticker in fitted_tickers])

    Sigma = nearest_psd(B @ F @ B.T + E)

    intercept_series = pd.Series(intercepts, name='raw_intercept').loc[fitted_tickers]
    common_intercept = float(intercept_series.mean())
    shrunk_intercepts = ((1.0 - residual_shrinkage) * intercept_series) + (residual_shrinkage * common_intercept)
    beta_df = pd.DataFrame(B, index=fitted_tickers, columns=FACTOR_COLS)
    beta_df['raw_intercept'] = intercept_series
    beta_df['shrunk_intercept'] = shrunk_intercepts
    beta_df['residual_var'] = pd.Series(residual_vars).loc[fitted_tickers]

    diagnostics = {
        'rebalance_date': pd.Timestamp(rebalance_date),
        'window_start': pd.Timestamp(window_dates.min()),
        'window_end': pd.Timestamp(window_dates.max()),
        'n_obs': int(len(window_dates)),
        'n_tickers': int(len(fitted_tickers)),
        'factor_shrinkage': float(factor_lw.shrinkage_),
        'residual_shrinkage': residual_shrinkage,
        'min_eigenvalue': float(np.linalg.eigvalsh(Sigma).min()),
    }
    return fitted_tickers, Sigma, beta_df, diagnostics


def solve_min_variance_weights(tickers, Sigma: np.ndarray) -> pd.Series:
    n = len(tickers)
    w = cp.Variable(n)
    problem = cp.Problem(
        cp.Minimize(cp.quad_form(w, cp.psd_wrap(Sigma))),
        [cp.sum(w) == 1.0, w >= MIN_WEIGHT, w <= MAX_WEIGHT],
    )
    for solver in ['CLARABEL', 'OSQP', 'SCS']:
        try:
            problem.solve(solver=solver, verbose=False)
        except Exception:
            continue
        if problem.status in ('optimal', 'optimal_inaccurate') and w.value is not None:
            weights = pd.Series(np.asarray(w.value).reshape(-1), index=tickers, dtype=float).clip(MIN_WEIGHT, MAX_WEIGHT)
            weights = weights / weights.sum()
            return weights
    raise RuntimeError(f'Min-var optimization failed: {problem.status}')


def project_long_only_capped_weights(raw_weights: pd.Series) -> pd.Series:
    tickers = list(raw_weights.index)
    raw = raw_weights.to_numpy(dtype=float)
    n = len(raw)
    w = cp.Variable(n)
    problem = cp.Problem(
        cp.Minimize(cp.sum_squares(w - raw)),
        [cp.sum(w) == 1.0, w >= MIN_WEIGHT, w <= MAX_WEIGHT],
    )
    for solver in ['OSQP', 'CLARABEL', 'SCS']:
        try:
            problem.solve(solver=solver, verbose=False)
        except Exception:
            continue
        if problem.status in ('optimal', 'optimal_inaccurate') and w.value is not None:
            weights = pd.Series(np.asarray(w.value).reshape(-1), index=tickers, dtype=float).clip(MIN_WEIGHT, MAX_WEIGHT)
            weights = weights / weights.sum()
            return weights
    clipped = raw_weights.clip(MIN_WEIGHT, MAX_WEIGHT)
    return clipped / clipped.sum()


def expand_weight_series(weight_series: pd.Series, all_tickers: list[str]) -> pd.Series:
    expanded = pd.Series(0.0, index=all_tickers, dtype=float)
    expanded.loc[weight_series.index] = weight_series.astype(float)
    return expanded

## 8. Rolling 10-Day Rebalance Loop

Each rebalance date uses only factor/return data and feature snapshots strictly before that date. The tilted portfolio is projected back to long-only, fully invested, capped weights.

In [ ]:
test_dates = pd.DatetimeIndex(adj_close.loc[test_start:test_end].index.unique()).sort_values()
rebalance_dates = test_dates[::REBALANCE_EVERY_DAYS]
print('Test trading dates:', len(test_dates))
print('Rebalance dates:', len(rebalance_dates))
print('First rebalance:', rebalance_dates[0])
print('Last rebalance:', rebalance_dates[-1])

feature_by_date = {pd.Timestamp(date): frame.copy() for date, frame in model_features.groupby('date', sort=True)}
available_feature_dates = pd.DatetimeIndex(sorted(feature_by_date.keys()))

minvar_rows = []
tilted_rows = []
xgb_rows = []
delta_rows = []
diagnostics_rows = []
last_beta_df = None

for idx, rebalance_date in enumerate(rebalance_dates, start=1):
    fitted_tickers, Sigma, beta_df, diagnostics = estimate_ew_ff5_covariance(rebalance_date)
    w_minvar = solve_min_variance_weights(fitted_tickers, Sigma)

    feature_pos = available_feature_dates.searchsorted(pd.Timestamp(rebalance_date), side='left') - 1
    if feature_pos < 0:
        raise ValueError(f'No feature snapshot before {rebalance_date}')
    signal_date = available_feature_dates[feature_pos]
    snapshot = feature_by_date[signal_date]
    snapshot = snapshot.loc[snapshot['ticker'].isin(fitted_tickers)].copy()

    proba = xgb_model.predict_proba(snapshot[TILT_FEATURE_NAMES])
    top_q_prob = pd.Series(proba[:, 4], index=snapshot['ticker'], name='top_quintile_probability')
    signal = top_q_prob.reindex(fitted_tickers).fillna(top_q_prob.mean())
    signal = signal - signal.mean()

    if float(signal.abs().sum()) > 0.0:
        delta = signal / signal.abs().sum() * TILT_ALPHA
    else:
        delta = signal * 0.0

    raw_tilted = w_minvar.add(delta, fill_value=0.0)
    w_tilted = project_long_only_capped_weights(raw_tilted)

    positive_scores = top_q_prob.reindex(fitted_tickers).clip(lower=0.0).fillna(0.0)
    if float(positive_scores.sum()) > 0.0:
        w_xgb = positive_scores / positive_scores.sum()
    else:
        w_xgb = pd.Series(1.0 / len(fitted_tickers), index=fitted_tickers)

    minvar_row = expand_weight_series(w_minvar, portfolio_tickers)
    tilted_row = expand_weight_series(w_tilted, portfolio_tickers)
    xgb_row = expand_weight_series(w_xgb, portfolio_tickers)
    delta_row = expand_weight_series(w_tilted - w_minvar, portfolio_tickers)

    minvar_row.name = pd.Timestamp(rebalance_date)
    tilted_row.name = pd.Timestamp(rebalance_date)
    xgb_row.name = pd.Timestamp(rebalance_date)
    delta_row.name = pd.Timestamp(rebalance_date)

    minvar_rows.append(minvar_row)
    tilted_rows.append(tilted_row)
    xgb_rows.append(xgb_row)
    delta_rows.append(delta_row)
    diagnostics_rows.append({**diagnostics, 'signal_date': pd.Timestamp(signal_date)})
    last_beta_df = beta_df

    if idx % 10 == 0 or idx == len(rebalance_dates):
        print(f'Processed {idx}/{len(rebalance_dates)} rebalances through {pd.Timestamp(rebalance_date).date()}')

minvar_weights = pd.DataFrame(minvar_rows)
tilted_weights = pd.DataFrame(tilted_rows)
xgb_weights = pd.DataFrame(xgb_rows)
tilt_deltas = pd.DataFrame(delta_rows)
risk_diagnostics = pd.DataFrame(diagnostics_rows)

minvar_weights.index.name = 'date'
tilted_weights.index.name = 'date'
xgb_weights.index.name = 'date'
tilt_deltas.index.name = 'date'

# Remove columns that are zero across every generated view, then validate row sums.
combined_abs_exposure = minvar_weights.abs().add(tilted_weights.abs(), fill_value=0.0).add(xgb_weights.abs(), fill_value=0.0)
active_columns = combined_abs_exposure.columns[(combined_abs_exposure.sum(axis=0) > 0.0)]
minvar_weights = minvar_weights.loc[:, active_columns]
tilted_weights = tilted_weights.loc[:, active_columns]
xgb_weights = xgb_weights.loc[:, active_columns]
tilt_deltas = tilt_deltas.loc[:, active_columns]

minvar_weights = validate_weights_frame(minvar_weights, dataset_name=DATASET_NAME, repo_root=repo_root)
tilted_weights = validate_weights_frame(tilted_weights, dataset_name=DATASET_NAME, repo_root=repo_root)
xgb_weights = validate_weights_frame(xgb_weights, dataset_name=DATASET_NAME, repo_root=repo_root)

print('Min-var weights shape:', minvar_weights.shape)
print('Tilted weights shape:', tilted_weights.shape)
print('Max tilted row-sum error:', float((tilted_weights.sum(axis=1) - 1.0).abs().max()))
print('Max single-name weight:', float(tilted_weights.max().max()))
display(risk_diagnostics.tail())

## 9. Inspect Final Optimal Portfolio Weights

The final row of the tilted weights frame is the final optimal portfolio weight vector produced by this notebook.

In [ ]:
final_rebalance_date = tilted_weights.index.max()
final_optimal_weights = tilted_weights.loc[final_rebalance_date].sort_values(ascending=False)
final_minvar_weights = minvar_weights.loc[final_rebalance_date]
final_delta = (tilted_weights.loc[final_rebalance_date] - minvar_weights.loc[final_rebalance_date]).sort_values()

print('Final rebalance date:', final_rebalance_date.date())
print('Final weight sum:', final_optimal_weights.sum())
print('Final active names:', int((final_optimal_weights > 0).sum()))
print('\nTop final optimal weights:')
print(final_optimal_weights[final_optimal_weights > 0].head(30).round(4).to_string())

print('\nLargest positive tilts:')
print(final_delta.tail(15).round(5).to_string())

print('\nLargest negative tilts:')
print(final_delta.head(15).round(5).to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
final_optimal_weights[final_optimal_weights > 0].head(30).sort_values().plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Final Tilted Portfolio Weights')
axes[0].set_xlabel('Weight')
final_delta.abs().sort_values(ascending=False).head(30).sort_values().plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Largest Absolute XGBoost Tilts')
axes[1].set_xlabel('Absolute delta')
plt.tight_layout()
plt.show()

## 10. Portfolio Objects And Backtest

Both the pure rolling minimum-variance baseline and the XGBoost-tilted strategy are backtested through the shared toolkit wrapper.

In [ ]:
def prepare_backtest_weight_frame(weights: pd.DataFrame) -> pd.DataFrame:
    prepared = weights.reindex(columns=portfolio_tickers, fill_value=0.0).astype(float)
    prepared = prepared.clip(lower=0.0, upper=MAX_WEIGHT)
    row_sums = prepared.sum(axis=1)
    if (row_sums <= 0.0).any():
        bad_dates = row_sums.loc[row_sums <= 0.0].index.tolist()
        raise ValueError(f'Cannot normalize zero-exposure weight rows: {bad_dates[:5]}')
    prepared = prepared.div(row_sums, axis=0)

    # Absorb tiny floating-point residuals into the largest position in each row so validation
    # and the toolkit-created equal-weight benchmark both see clean, fully invested rows.
    for date_value in prepared.index:
        residual = 1.0 - float(prepared.loc[date_value].sum())
        if abs(residual) <= 1e-12:
            continue
        target_ticker = prepared.loc[date_value].idxmax()
        prepared.loc[date_value, target_ticker] += residual

    return validate_weights_frame(prepared, dataset_name=DATASET_NAME, repo_root=repo_root)


minvar_weights_for_backtest = prepare_backtest_weight_frame(minvar_weights)
tilted_weights_for_backtest = prepare_backtest_weight_frame(tilted_weights)

portfolio_minvar = PortfolioWeights(
    weights=minvar_weights_for_backtest,
    dataset_name=DATASET_NAME,
    strategy_name=MINVAR_MODEL_NAME,
    metadata={
        'type': 'rolling_minvar',
        'factor_model': 'ew_ff5',
        'prediction_horizon_days': PREDICTION_HORIZON_DAYS,
        'rebalance_every_days': REBALANCE_EVERY_DAYS,
        'ff_window': FF_WINDOW,
        'ew_halflife': EW_HALFLIFE,
    },
)

portfolio_tilted = PortfolioWeights(
    weights=tilted_weights_for_backtest,
    dataset_name=DATASET_NAME,
    strategy_name=MODEL_NAME,
    metadata={
        'type': 'rolling_minvar_xgb_tilt',
        'factor_model': 'ew_ff5',
        'prediction_horizon_days': PREDICTION_HORIZON_DAYS,
        'rebalance_every_days': REBALANCE_EVERY_DAYS,
        'ff_window': FF_WINDOW,
        'ew_halflife': EW_HALFLIFE,
        'tilt_alpha': TILT_ALPHA,
        'iv_source': 'current_yfinance_option_chain_static_by_ticker',
    },
)

print('Min-var backtest row-sum max error:', float((portfolio_minvar.weights.sum(axis=1) - 1.0).abs().max()))
print('Tilted backtest row-sum max error:', float((portfolio_tilted.weights.sum(axis=1) - 1.0).abs().max()))
print('Backtest ticker columns:', len(portfolio_tilted.weights.columns), 'of', len(portfolio_tickers))

result_minvar = backtest_weights(DATASET_NAME, portfolio_minvar, repo_root=repo_root)
result_tilted = backtest_weights(DATASET_NAME, portfolio_tilted, repo_root=repo_root)

metrics_minvar = build_metrics(result_minvar)
metrics_tilted = build_metrics(result_tilted)

artifact_paths_minvar = write_backtest_artifacts(result_minvar, OUTPUT_DIR / 'minvar')
artifact_paths_tilted = write_backtest_artifacts(result_tilted, OUTPUT_DIR / 'tilted')

metrics_df = pd.DataFrame({'Min-Var EW FF5': metrics_minvar, 'Min-Var + XGB Tilt': metrics_tilted}).T
print('Test-period metrics:')
display(metrics_df)
print('Tilted QuantStats report:', artifact_paths_tilted['quantstats_report'])

## 11. Backtest Plots

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
result_minvar.nav.rename('Min-Var EW FF5').plot(ax=ax, color='steelblue', linewidth=1.8)
result_tilted.nav.rename('Min-Var + XGB Tilt').plot(ax=ax, color='coral', linewidth=1.8)
if 'SPY' in result_tilted.benchmark_returns.columns:
    spy_nav = (1.0 + result_tilted.benchmark_returns['SPY']).cumprod() * result_tilted.nav.iloc[0]
    spy_nav.rename('SPY').plot(ax=ax, linestyle='--', color='gray', linewidth=1.4)
ax.set_title('Rolling 10-Day Portfolio NAV')
ax.set_ylabel('NAV')
ax.legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for result, label, color in [
    (result_minvar, 'Min-Var EW FF5', 'steelblue'),
    (result_tilted, 'Min-Var + XGB Tilt', 'coral'),
]:
    drawdown = result.nav / result.nav.cummax() - 1.0
    result.returns.rename(label).plot(ax=axes[0], color=color, alpha=0.8, linewidth=0.8)
    drawdown.rename(label).plot(ax=axes[1], color=color, linewidth=1.2)
axes[0].set_title('Daily Returns')
axes[0].axhline(0, color='black', linewidth=0.5)
axes[0].legend()
axes[1].set_title('Drawdown')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].legend()
plt.tight_layout()
plt.show()

## 12. Save Model And Log To MLflow

The MLflow run logs the tilted portfolio, backtest artifacts, XGBoost JSON model, and the configuration needed to understand the run.

In [ ]:
xgb_model.save_model(MODEL_ARTIFACT_PATH)

mlflow_layout = init_mlflow(repo_root)
print('MLflow tracking URI:', mlflow_layout['tracking_uri'])

with start_run(
    run_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    tags={
        'model_family': 'xgboost',
        'strategy_type': 'rolling_minvar_xgb_tilt',
        'factor_model': 'ew_ff5',
        'prediction_horizon_days': str(PREDICTION_HORIZON_DAYS),
        'rebalance_every_days': str(REBALANCE_EVERY_DAYS),
    },
    repo_root=repo_root,
):
    mlflow.log_params(
        {
            'dataset_name': DATASET_NAME,
            'prediction_horizon_days': PREDICTION_HORIZON_DAYS,
            'rebalance_every_days': REBALANCE_EVERY_DAYS,
            'ff_window': FF_WINDOW,
            'ew_halflife': EW_HALFLIFE,
            'tilt_alpha': TILT_ALPHA,
            'min_weight': MIN_WEIGHT,
            'max_weight': MAX_WEIGHT,
            'xgb_best_iteration': int(getattr(xgb_model, 'best_iteration', -1) or -1),
            'xgb_val_accuracy': float(val_accuracy),
            'xgb_val_log_loss': float(val_log_loss),
            'feature_count': len(TILT_FEATURE_NAMES),
            'feature_names': ','.join(TILT_FEATURE_NAMES),
            'iv_source': 'current_yfinance_option_chain_static_by_ticker',
            'iv_missing_count': int(iv_snapshot['current_iv_missing'].sum()),
            'rebalance_count': len(tilted_weights),
            'active_ticker_count': len(tilted_weights.columns),
        }
    )
    log_portfolio(portfolio_tilted)
    log_backtest(result_tilted)
    manifest = log_model_submission(
        {'model': MODEL_ARTIFACT_PATH},
        model_name=MODEL_NAME,
        model_family='xgboost',
        feature_names=TILT_FEATURE_NAMES,
        target=target_col,
        horizon=PREDICTION_HORIZON_DAYS,
        rebalance_frequency=f'every_{REBALANCE_EVERY_DAYS}_trading_days',
        preprocessing={
            'cross_sectional_rank_normalized': rank_normalized_features,
            'binary_features': ['current_iv_missing'],
            'iv_source': 'current_yfinance_option_chain_static_by_ticker',
        },
        model_config={
            'library': 'xgboost',
            'estimator': 'XGBClassifier',
            'objective': 'multi:softprob',
            'num_class': 5,
            'factor_model': 'ew_ff5',
            'ff_window': FF_WINDOW,
            'ew_halflife': EW_HALFLIFE,
            'tilt_alpha': TILT_ALPHA,
            'portfolio_builder': 'notebook_local_minvar_plus_centered_xgb_tilt',
        },
        source_files=[repo_root / 'MODELS' / 'Eamon' / 'eamon_xgboost_final.ipynb'],
        notes='Rolling 10-day min-variance portfolio using EW FF5 betas and a small XGBoost top-quintile tilt.',
    )

print('Model artifact:', MODEL_ARTIFACT_PATH)
print('MLflow logging complete.')


## Four-Regime Proxy Dataset Backtests

This section is self-contained: it bootstraps the repo, rebuilds Eamon's feature pipeline, downloads the saved `eamon_xgboost_final` XGBoost model from MLflow, runs the four proxy-regime backtests, summarizes the `$4.8M` bankroll, and logs the results back to MLflow. You can run these cells from a fresh kernel without running the training cells above.

Set `EAMON_FINAL_MLFLOW_RUN_ID` or `EAMON_FINAL_MLFLOW_ARTIFACT_PATH` before running if you want to force a specific MLflow run or model artifact.

In [1]:
import json
import os
import sys
import warnings
from datetime import timedelta
from pathlib import Path

warnings.filterwarnings("ignore")

import cvxpy as cp
import mlflow
import numpy as np
import pandas as pd
import pandas_datareader.data as pdr
import xgboost as xgb
import yfinance as yf
from IPython.display import display
from mlflow.tracking import MlflowClient
from sklearn.covariance import LedoitWolf


def _eamon_final_find_repo_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "configs" / "datasets.toml").exists() and (candidate / "src" / "portfolio_toolkit").exists():
            return candidate
    raise RuntimeError("Could not locate the Portfolio-Optimizer repo root from the current working directory.")


repo_root = _eamon_final_find_repo_root()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from portfolio_toolkit import (  # noqa: E402
    PortfolioWeights,
    backtest_weights,
    build_features,
    build_metrics,
    get_dataset_spec,
    init_mlflow,
    load_prices,
    log_backtest,
    log_portfolio,
    start_run,
    validate_weights_frame,
    write_backtest_artifacts,
)

MODEL_NAME = "eamon_xgboost_final"
MINVAR_MODEL_NAME = "eamon_xgboost_final_minvar"
PREDICTION_HORIZON_DAYS = 20
REBALANCE_EVERY_DAYS = 10
FF_WINDOW = 252
EW_HALFLIFE = 63
TILT_ALPHA = 0.25
MIN_WEIGHT = 0.0
MAX_WEIGHT = 0.15
MIN_FACTOR_OBS = 126
FACTOR_COLS = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]

TILT_FEATURE_NAMES = [
    "cs_rank_vol_60d",
    "downside_vol_20d",
    "cs_rank_beta_60d_spy",
    "spy_return_60d",
    "vol_10d",
    "return_dispersion_20d",
    "spy_vol_60d",
    "vol_5d",
    "spy_vol_20d",
    "price_to_sma_200d",
    "cs_rank_distance_to_60d_high",
    "cs_rank_momentum_120d",
    "vol_of_vol_60d",
    "price_to_sma_50d",
    "current_iv",
    "price_to_sma_20d",
    "vol_20d",
    "atr_14",
    "volume_zscore_20d",
    "mom_60_20_divergence",
    "dollar_volume_ratio_20d",
    "vol_of_vol_20d",
    "upside_vol_20d",
    "price_to_sma_10d",
    "vol_of_vol_10d",
    "current_iv_missing",
    "volume_zscore_60d",
    "volume_change_5d",
    "volume_change_1d",
    "volume_change_20d",
]

TOOLKIT_FEATURE_NAMES = [
    "downside_vol_20d",
    "vol_10d",
    "vol_5d",
    "price_to_sma_200d",
    "price_to_sma_50d",
    "price_to_sma_20d",
    "vol_20d",
    "atr_14",
    "volume_zscore_20d",
    "dollar_volume_ratio_20d",
    "upside_vol_20d",
    "price_to_sma_10d",
    "volume_zscore_60d",
    "volume_change_5d",
    "volume_change_1d",
    "volume_change_20d",
    "vol_60d",
    "beta_60d_spy",
    "distance_to_60d_high",
    "momentum_120d",
    "momentum_60d",
    "momentum_20d",
]

if len(TILT_FEATURE_NAMES) != len(set(TILT_FEATURE_NAMES)):
    raise ValueError("Duplicate feature names found in TILT_FEATURE_NAMES.")

EAMON_FINAL_REGIME_BACKTEST_DATASETS = [
    "regime_modern_tech_gain_2022_2026",
    "regime_financial_crisis_loss_2005_2010",
    "regime_nineties_volatility_1995_1999",
    "regime_oil_pre2014_energy_2010_2013",
]
EAMON_FINAL_REGIME_BACKTEST_ALLOCATION = 1_200_000.0
EAMON_FINAL_REGIME_BACKTEST_TOTAL_BANKROLL = EAMON_FINAL_REGIME_BACKTEST_ALLOCATION * len(EAMON_FINAL_REGIME_BACKTEST_DATASETS)
EAMON_FINAL_REGIME_BACKTEST_OUTPUT_DIR = repo_root / "runs" / "eamon_xgboost_final_four_regime_proxy_backtests"
EAMON_FINAL_REGIME_BACKTEST_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EAMON_FINAL_FOUR_REGIME_LOG_TO_MLFLOW = os.environ.get("EAMON_FINAL_FOUR_REGIME_LOG_TO_MLFLOW", "1") == "1"
EAMON_FINAL_REFRESH_IV = os.environ.get("EAMON_FINAL_REFRESH_IV", "0") == "1"

print("Repo root:", repo_root)
print("Backtest datasets:", EAMON_FINAL_REGIME_BACKTEST_DATASETS)
print("Model source: MLflow submission artifact for", MODEL_NAME)
print(f"Bankroll: ${EAMON_FINAL_REGIME_BACKTEST_TOTAL_BANKROLL:,.0f} total, ${EAMON_FINAL_REGIME_BACKTEST_ALLOCATION:,.0f} per dataset")


Repo root: /Users/adamthorne/Desktop/Portfolio/Portfolio-Optimizer
Backtest datasets: ['regime_modern_tech_gain_2022_2026', 'regime_financial_crisis_loss_2005_2010', 'regime_nineties_volatility_1995_1999', 'regime_oil_pre2014_energy_2010_2013']
Model source: MLflow submission artifact for eamon_xgboost_final
Bankroll: $4,800,000 total, $1,200,000 per dataset


In [2]:
def _eamon_final_sort_runs_latest_first(runs: pd.DataFrame) -> pd.DataFrame:
    if runs.empty:
        return runs
    sorted_runs = runs.copy()
    if "start_time" in sorted_runs.columns:
        sorted_runs = sorted_runs.sort_values("start_time", ascending=False)
    return sorted_runs.reset_index(drop=True)


def _eamon_final_find_latest_mlflow_run_id(model_name: str = MODEL_NAME) -> str:
    forced_run_id = os.environ.get("EAMON_FINAL_MLFLOW_RUN_ID", "").strip()
    if forced_run_id:
        return forced_run_id

    init_mlflow(repo_root=repo_root)
    experiments = mlflow.search_experiments()
    experiment_ids = [experiment.experiment_id for experiment in experiments]
    if not experiment_ids:
        raise RuntimeError("MLflow has no experiments visible to search for Eamon's model submission.")

    filter_strings = [
        f"params.submission_model_name = '{model_name}'",
        f"attributes.run_name = '{model_name}'",
        f"params.model_name = '{model_name}'",
    ]
    for filter_string in filter_strings:
        try:
            runs = mlflow.search_runs(
                experiment_ids=experiment_ids,
                filter_string=filter_string,
                max_results=200,
            )
        except Exception as exc:
            print(f"MLflow filtered run search failed for {filter_string!r}: {exc}")
            runs = pd.DataFrame()
        runs = _eamon_final_sort_runs_latest_first(runs)
        if not runs.empty:
            return str(runs.loc[0, "run_id"])

    runs = mlflow.search_runs(experiment_ids=experiment_ids, max_results=1000)
    if runs.empty:
        raise RuntimeError("MLflow search returned no runs.")

    mask = pd.Series(False, index=runs.index)
    for column in ["tags.mlflow.runName", "params.submission_model_name", "params.model_name"]:
        if column in runs.columns:
            mask = mask | (runs[column].astype(str) == model_name)
    candidates = _eamon_final_sort_runs_latest_first(runs.loc[mask])
    if candidates.empty:
        raise RuntimeError(
            f"Could not find an MLflow run for {model_name!r}. "
            "Set EAMON_FINAL_MLFLOW_RUN_ID to the exact run id if the run has a different name."
        )
    return str(candidates.loc[0, "run_id"])


def _eamon_final_list_artifacts_recursive(client: MlflowClient, run_id: str, path: str = "") -> list[str]:
    artifact_paths: list[str] = []
    for artifact in client.list_artifacts(run_id, path):
        if artifact.is_dir:
            artifact_paths.extend(_eamon_final_list_artifacts_recursive(client, run_id, artifact.path))
        else:
            artifact_paths.append(artifact.path)
    return artifact_paths


def _eamon_final_model_artifact_path_from_manifest(run_id: str, manifest_path: str) -> str:
    local_manifest_path = Path(mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=manifest_path))
    manifest = json.loads(local_manifest_path.read_text(encoding="utf-8"))
    artifact_map = manifest.get("artifact_map", {})
    artifact_files = manifest.get("artifact_files", [])
    relative_model_path = artifact_map.get("model")
    if relative_model_path is None:
        json_files = [path for path in artifact_files if str(path).endswith(".json")]
        if not json_files:
            raise RuntimeError(f"Manifest {manifest_path} does not contain a JSON model artifact.")
        relative_model_path = json_files[0]
    manifest_dir = Path(manifest_path).parent
    if str(manifest_dir) == ".":
        return str(Path(relative_model_path)).replace("\\", "/")
    return str(manifest_dir / relative_model_path).replace("\\", "/")


def _eamon_final_choose_model_artifact_path(run_id: str) -> str:
    forced_artifact_path = os.environ.get("EAMON_FINAL_MLFLOW_ARTIFACT_PATH", "").strip()
    if forced_artifact_path:
        return forced_artifact_path

    client = MlflowClient()
    artifact_paths = _eamon_final_list_artifacts_recursive(client, run_id)
    manifest_paths = [path for path in artifact_paths if path.endswith("model_submission/manifest.json")]
    if manifest_paths:
        return _eamon_final_model_artifact_path_from_manifest(run_id, manifest_paths[0])

    json_candidates = [
        path
        for path in artifact_paths
        if path.endswith(".json") and not path.endswith("manifest.json") and MODEL_NAME in Path(path).name
    ]
    if not json_candidates:
        json_candidates = [path for path in artifact_paths if path.endswith(".json") and not path.endswith("manifest.json")]
    if not json_candidates:
        raise RuntimeError(
            f"Run {run_id} has no JSON model artifact. "
            "Set EAMON_FINAL_MLFLOW_ARTIFACT_PATH if the artifact was logged under a custom path."
        )
    json_candidates = sorted(json_candidates, key=lambda path: ("model_submission/artifacts" not in path, len(path)))
    return json_candidates[0]


def _eamon_final_load_xgb_model_from_mlflow() -> tuple[xgb.XGBClassifier, dict[str, str]]:
    init_mlflow(repo_root=repo_root)
    run_id = _eamon_final_find_latest_mlflow_run_id(MODEL_NAME)
    artifact_path = _eamon_final_choose_model_artifact_path(run_id)
    local_model_path = Path(mlflow.artifacts.download_artifacts(run_id=run_id, artifact_path=artifact_path))
    model = xgb.XGBClassifier()
    model.load_model(local_model_path)
    model_info = {
        "run_id": run_id,
        "artifact_path": artifact_path,
        "local_model_path": str(local_model_path),
    }
    print("Loaded Eamon XGBoost model from MLflow:")
    print(json.dumps(model_info, indent=2))
    return model, model_info


In [3]:
def _eamon_final_normalize_extra_yfinance_frame(frame: pd.DataFrame, ticker: str) -> pd.DataFrame:
    normalized = frame.copy()
    if isinstance(normalized.columns, pd.MultiIndex):
        normalized.columns = normalized.columns.get_level_values(0)
    normalized.columns.name = None
    if "Adj Close" not in normalized.columns and "Close" in normalized.columns:
        normalized["Adj Close"] = normalized["Close"]
    normalized = normalized.reset_index().rename(
        columns={
            "Date": "date",
            "Datetime": "date",
            "Open": "open",
            "High": "high",
            "Low": "low",
            "Close": "close",
            "Adj Close": "adj_close",
            "Volume": "volume",
        }
    )
    normalized["ticker"] = ticker.upper()
    normalized = normalized.loc[:, ["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]]
    normalized["date"] = pd.to_datetime(normalized["date"], utc=True).dt.tz_localize(None)
    for column in ["open", "high", "low", "close", "adj_close", "volume"]:
        normalized[column] = pd.to_numeric(normalized[column], errors="coerce")
    return normalized.dropna(subset=["date", "adj_close"]).sort_values(["ticker", "date"]).reset_index(drop=True)


def _eamon_final_download_context_ticker(ticker: str, spec_obj) -> pd.DataFrame:
    start = pd.Timestamp(spec_obj.start_date).date().isoformat()
    end = (pd.Timestamp(spec_obj.end_date).date() + timedelta(days=1)).isoformat()
    downloaded = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=False,
        progress=False,
        threads=False,
    )
    if downloaded.empty:
        raise ValueError(f"No yfinance rows downloaded for required context ticker {ticker} in {spec_obj.identifier}.")
    return _eamon_final_normalize_extra_yfinance_frame(downloaded, ticker)


def _eamon_final_prices_with_spy_context(prices_frame: pd.DataFrame, spec_obj) -> pd.DataFrame:
    prices_for_features = prices_frame.copy()
    prices_for_features["date"] = pd.to_datetime(prices_for_features["date"], utc=True).dt.tz_localize(None)
    prices_for_features["ticker"] = prices_for_features["ticker"].astype(str).str.upper()
    if "SPY" not in set(prices_for_features["ticker"]):
        spy_context = _eamon_final_download_context_ticker("SPY", spec_obj)
        prices_for_features = pd.concat([prices_for_features, spy_context], ignore_index=True)
    return prices_for_features.sort_values(["ticker", "date"]).reset_index(drop=True)


def _eamon_final_nearest_atm_iv(option_frame: pd.DataFrame, spot: float) -> float:
    if option_frame.empty or not np.isfinite(spot):
        return np.nan
    frame = option_frame.loc[:, ["strike", "impliedVolatility"]].copy()
    frame["impliedVolatility"] = pd.to_numeric(frame["impliedVolatility"], errors="coerce")
    frame = frame.dropna(subset=["strike", "impliedVolatility"])
    if frame.empty:
        return np.nan
    idx = (frame["strike"] - spot).abs().idxmin()
    return float(frame.loc[idx, "impliedVolatility"])


def _eamon_final_fetch_current_iv_snapshot(tickers: list[str], latest_prices: pd.Series, cache_path: Path, refresh: bool = False) -> pd.DataFrame:
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    if cache_path.exists() and not refresh:
        cached = pd.read_csv(cache_path)
        cached["ticker"] = cached["ticker"].astype(str).str.upper()
        if "current_iv" not in cached.columns:
            raw = cached.get("current_iv_raw", pd.Series(np.nan, index=cached.index))
            median_iv = raw.median(skipna=True)
            if not np.isfinite(median_iv):
                median_iv = 0.25
            cached["current_iv_missing"] = raw.isna().astype(float)
            cached["current_iv"] = raw.fillna(median_iv)
        return cached

    rows = []
    for idx, ticker in enumerate(tickers, start=1):
        spot = float(latest_prices.get(ticker, np.nan))
        current_iv = np.nan
        expiration = None
        try:
            yf_ticker = yf.Ticker(ticker)
            expirations = list(yf_ticker.options)
            if expirations:
                expiration = expirations[0]
                chain = yf_ticker.option_chain(expiration)
                call_iv = _eamon_final_nearest_atm_iv(chain.calls, spot)
                put_iv = _eamon_final_nearest_atm_iv(chain.puts, spot)
                current_iv = float(np.nanmean([call_iv, put_iv]))
        except Exception as exc:
            print(f"IV fetch failed for {ticker}: {exc}")
        rows.append({"ticker": ticker, "current_iv_raw": current_iv, "iv_expiration": expiration, "iv_spot": spot})
        if idx % 25 == 0 or idx == len(tickers):
            print(f"Fetched IV for {idx}/{len(tickers)} tickers")

    iv = pd.DataFrame(rows)
    median_iv = iv["current_iv_raw"].median(skipna=True)
    if not np.isfinite(median_iv):
        median_iv = 0.25
    iv["current_iv_missing"] = iv["current_iv_raw"].isna().astype(float)
    iv["current_iv"] = iv["current_iv_raw"].fillna(median_iv)
    iv.to_csv(cache_path, index=False)
    return iv


def _eamon_final_add_vol_of_vol_features(feature_frame: pd.DataFrame, prices_frame: pd.DataFrame) -> pd.DataFrame:
    panel = prices_frame.sort_values(["ticker", "date"]).loc[:, ["date", "ticker", "adj_close"]].copy()
    panel["return_1d_for_vov"] = panel.groupby("ticker")["adj_close"].pct_change()
    short_vol = panel.groupby("ticker")["return_1d_for_vov"].transform(lambda s: s.rolling(5, min_periods=5).std(ddof=0))
    vov = panel.loc[:, ["date", "ticker"]].copy()
    for window in [10, 20, 60]:
        vov[f"vol_of_vol_{window}d"] = short_vol.groupby(panel["ticker"]).transform(
            lambda s, window=window: s.rolling(window, min_periods=window).std(ddof=0)
        )
    return feature_frame.merge(vov, on=["date", "ticker"], how="left")


def _eamon_final_build_market_context(prices_frame: pd.DataFrame) -> pd.DataFrame:
    panel = prices_frame.sort_values(["ticker", "date"]).copy()
    panel["return_1d_local"] = panel.groupby("ticker", sort=False)["adj_close"].pct_change()
    wide_returns = panel.pivot(index="date", columns="ticker", values="return_1d_local").sort_index()
    spy = (
        panel.loc[panel["ticker"] == "SPY", ["date", "adj_close"]]
        .drop_duplicates("date")
        .set_index("date")["adj_close"]
        .sort_index()
    )
    if spy.empty:
        raise ValueError("SPY context is required for Eamon's market context features.")
    market = pd.DataFrame(index=spy.index)
    market["spy_return_60d"] = spy.pct_change(60)
    spy_daily = spy.pct_change()
    market["spy_vol_20d"] = spy_daily.rolling(20, min_periods=20).std(ddof=0)
    market["spy_vol_60d"] = spy_daily.rolling(60, min_periods=60).std(ddof=0)
    tradable_cols = [column for column in wide_returns.columns if column != "SPY"]
    market["return_dispersion_20d"] = wide_returns[tradable_cols].std(axis=1, skipna=True).rolling(20, min_periods=20).mean()
    return market.reset_index().rename(columns={"index": "date"})


def _eamon_final_add_custom_features(features: pd.DataFrame) -> pd.DataFrame:
    features = features.copy()
    features["mom_60_20_divergence"] = features["momentum_60d"] - features["momentum_20d"]
    for name in ["vol_60d", "beta_60d_spy", "distance_to_60d_high", "momentum_120d"]:
        features[f"cs_rank_{name}"] = features.groupby("date")[name].rank(pct=True, method="average")
    return features


def _eamon_final_build_model_features(prices_frame: pd.DataFrame, tradable_tickers: list[str], iv_snapshot: pd.DataFrame) -> pd.DataFrame:
    feature_frame = build_features(prices_frame, feature_names=TOOLKIT_FEATURE_NAMES)
    feature_frame = _eamon_final_add_vol_of_vol_features(feature_frame, prices_frame)
    feature_frame = feature_frame.merge(_eamon_final_build_market_context(prices_frame), on="date", how="left")
    feature_frame = _eamon_final_add_custom_features(feature_frame)
    feature_frame = feature_frame.merge(
        iv_snapshot.loc[:, ["ticker", "current_iv", "current_iv_missing"]],
        on="ticker",
        how="left",
    )
    median_iv = feature_frame["current_iv"].median(skipna=True)
    if not np.isfinite(median_iv):
        median_iv = 0.25
    feature_frame["current_iv_missing"] = feature_frame["current_iv_missing"].fillna(1.0)
    feature_frame["current_iv"] = feature_frame["current_iv"].fillna(median_iv)

    missing_tilt_features = [feature for feature in TILT_FEATURE_NAMES if feature not in feature_frame.columns]
    if missing_tilt_features:
        raise KeyError(f"TILT_FEATURE_NAMES missing from feature_frame: {missing_tilt_features}")

    tradable_set = {ticker.upper() for ticker in tradable_tickers}
    model_features = (
        feature_frame.loc[feature_frame["ticker"].isin(tradable_set), ["date", "ticker"] + TILT_FEATURE_NAMES]
        .replace([np.inf, -np.inf], np.nan)
        .dropna(subset=TILT_FEATURE_NAMES)
        .sort_values(["date", "ticker"])
        .reset_index(drop=True)
    )
    rank_normalized_features = [feature for feature in TILT_FEATURE_NAMES if feature != "current_iv_missing"]
    for feature in rank_normalized_features:
        model_features[feature] = model_features.groupby("date")[feature].transform(lambda x: x.rank(pct=True))
    return model_features


In [4]:
def _eamon_final_exponential_sample_weights(n: int, halflife: float) -> np.ndarray:
    ages = np.arange(n - 1, -1, -1, dtype=float)
    weights = 0.5 ** (ages / float(halflife))
    return weights / weights.mean()


def _eamon_final_weighted_factor_regression(y: np.ndarray, X: np.ndarray, weights: np.ndarray) -> tuple[float, np.ndarray, np.ndarray]:
    x_design = np.column_stack([np.ones(len(X)), X])
    sqrt_w = np.sqrt(weights)
    beta, *_ = np.linalg.lstsq(x_design * sqrt_w[:, None], y * sqrt_w, rcond=None)
    fitted = x_design @ beta
    residuals = y - fitted
    return float(beta[0]), beta[1:].astype(float), residuals.astype(float)


def _eamon_final_nearest_psd(matrix: np.ndarray, jitter: float = 1e-8) -> np.ndarray:
    sym = (matrix + matrix.T) / 2.0
    min_eig = float(np.linalg.eigvalsh(sym).min())
    if min_eig < jitter:
        sym = sym + np.eye(sym.shape[0]) * (abs(min_eig) + jitter)
    return sym


def _eamon_final_load_ff5_for_prices(prices_frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    ff_start = pd.to_datetime(prices_frame["date"]).min().date().isoformat()
    ff_end = pd.to_datetime(prices_frame["date"]).max().date().isoformat()
    print("Downloading FF5 daily factors:", ff_start, "->", ff_end)
    ff5_raw = pdr.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench", start=ff_start, end=ff_end)[0]
    ff5 = ff5_raw / 100.0
    ff5.index = pd.to_datetime(ff5.index)
    return ff5[FACTOR_COLS].copy(), ff5["RF"].copy()


def _eamon_final_estimate_ew_ff5_covariance(
    rebalance_date: pd.Timestamp,
    returns_wide_local: pd.DataFrame,
    portfolio_tickers_local: list[str],
    factors_local: pd.DataFrame,
    rf_local: pd.Series,
    common_factor_dates_local: pd.DatetimeIndex,
) -> tuple[list[str], np.ndarray, pd.DataFrame, dict[str, object]]:
    eligible_dates = common_factor_dates_local[common_factor_dates_local < pd.Timestamp(rebalance_date)]
    window_dates = eligible_dates[-FF_WINDOW:]
    if len(window_dates) < MIN_FACTOR_OBS:
        raise ValueError(f"Not enough factor observations before {rebalance_date}: {len(window_dates)}")

    window_returns = returns_wide_local.loc[window_dates, portfolio_tickers_local]
    window_factors = factors_local.loc[window_dates, FACTOR_COLS]
    window_rf = rf_local.loc[window_dates]
    beta_rows = {}
    intercepts = {}
    residual_series = {}
    residual_vars = {}

    for ticker in portfolio_tickers_local:
        y_raw = window_returns[ticker] - window_rf
        valid = y_raw.notna() & window_factors.notna().all(axis=1) & window_rf.notna()
        if int(valid.sum()) < MIN_FACTOR_OBS:
            continue
        y = y_raw.loc[valid].to_numpy(dtype=float)
        X = window_factors.loc[valid, FACTOR_COLS].to_numpy(dtype=float)
        sample_weights = _eamon_final_exponential_sample_weights(len(y), EW_HALFLIFE)
        intercept, beta, residuals = _eamon_final_weighted_factor_regression(y, X, sample_weights)
        beta_rows[ticker] = beta
        intercepts[ticker] = intercept
        residual_series[ticker] = pd.Series(residuals, index=window_factors.loc[valid].index)
        residual_vars[ticker] = float(np.average(residuals ** 2, weights=sample_weights))

    fitted_tickers = list(beta_rows.keys())
    if len(fitted_tickers) < max(8, int(1.0 / MAX_WEIGHT) + 1):
        raise ValueError(f"Only {len(fitted_tickers)} tickers fitted before {rebalance_date}")

    B = np.vstack([beta_rows[ticker] for ticker in fitted_tickers])
    factor_lw = LedoitWolf().fit(window_factors.loc[:, FACTOR_COLS].dropna().to_numpy(dtype=float))
    F = factor_lw.covariance_

    residual_frame = pd.DataFrame(residual_series).loc[:, fitted_tickers]
    residual_centered = residual_frame - residual_frame.mean(skipna=True)
    residual_for_lw = residual_centered.fillna(0.0)
    residual_shrinkage = 1.0
    if residual_for_lw.shape[0] >= 20 and residual_for_lw.shape[1] >= 2:
        residual_lw = LedoitWolf().fit(residual_for_lw.to_numpy(dtype=float))
        E = residual_lw.covariance_
        residual_shrinkage = float(residual_lw.shrinkage_)
    else:
        E = np.diag([residual_vars[ticker] for ticker in fitted_tickers])

    Sigma = _eamon_final_nearest_psd(B @ F @ B.T + E)
    intercept_series = pd.Series(intercepts, name="raw_intercept").loc[fitted_tickers]
    common_intercept = float(intercept_series.mean())
    shrunk_intercepts = ((1.0 - residual_shrinkage) * intercept_series) + (residual_shrinkage * common_intercept)
    beta_df = pd.DataFrame(B, index=fitted_tickers, columns=FACTOR_COLS)
    beta_df["raw_intercept"] = intercept_series
    beta_df["shrunk_intercept"] = shrunk_intercepts
    beta_df["residual_var"] = pd.Series(residual_vars).loc[fitted_tickers]
    diagnostics = {
        "rebalance_date": pd.Timestamp(rebalance_date),
        "window_start": pd.Timestamp(window_dates.min()),
        "window_end": pd.Timestamp(window_dates.max()),
        "n_obs": int(len(window_dates)),
        "n_tickers": int(len(fitted_tickers)),
        "factor_shrinkage": float(factor_lw.shrinkage_),
        "residual_shrinkage": residual_shrinkage,
        "min_eigenvalue": float(np.linalg.eigvalsh(Sigma).min()),
    }
    return fitted_tickers, Sigma, beta_df, diagnostics


def _eamon_final_solve_min_variance_weights(tickers: list[str], sigma: np.ndarray) -> pd.Series:
    n = len(tickers)
    w = cp.Variable(n)
    problem = cp.Problem(
        cp.Minimize(cp.quad_form(w, cp.psd_wrap(sigma))),
        [cp.sum(w) == 1.0, w >= MIN_WEIGHT, w <= MAX_WEIGHT],
    )
    for solver in ["CLARABEL", "OSQP", "SCS"]:
        try:
            problem.solve(solver=solver, verbose=False)
        except Exception:
            continue
        if problem.status in ("optimal", "optimal_inaccurate") and w.value is not None:
            weights = pd.Series(np.asarray(w.value).reshape(-1), index=tickers, dtype=float).clip(MIN_WEIGHT, MAX_WEIGHT)
            return weights / weights.sum()
    raise RuntimeError(f"Min-var optimization failed: {problem.status}")


def _eamon_final_project_long_only_capped_weights(raw_weights: pd.Series) -> pd.Series:
    tickers = list(raw_weights.index)
    raw = raw_weights.to_numpy(dtype=float)
    w = cp.Variable(len(raw))
    problem = cp.Problem(
        cp.Minimize(cp.sum_squares(w - raw)),
        [cp.sum(w) == 1.0, w >= MIN_WEIGHT, w <= MAX_WEIGHT],
    )
    for solver in ["OSQP", "CLARABEL", "SCS"]:
        try:
            problem.solve(solver=solver, verbose=False)
        except Exception:
            continue
        if problem.status in ("optimal", "optimal_inaccurate") and w.value is not None:
            weights = pd.Series(np.asarray(w.value).reshape(-1), index=tickers, dtype=float).clip(MIN_WEIGHT, MAX_WEIGHT)
            return weights / weights.sum()
    clipped = raw_weights.clip(MIN_WEIGHT, MAX_WEIGHT)
    return clipped / clipped.sum()


def _eamon_final_expand_weight_series(weight_series: pd.Series, all_tickers: list[str]) -> pd.Series:
    expanded = pd.Series(0.0, index=all_tickers, dtype=float)
    expanded.loc[weight_series.index] = weight_series.astype(float)
    return expanded


def _eamon_final_prepare_backtest_weight_frame(weights: pd.DataFrame, tradable_tickers: list[str], spec_obj) -> pd.DataFrame:
    prepared = weights.reindex(columns=tradable_tickers, fill_value=0.0).astype(float)
    prepared = prepared.clip(lower=0.0, upper=MAX_WEIGHT)
    row_sums = prepared.sum(axis=1)
    if (row_sums <= 0.0).any():
        bad_dates = row_sums.loc[row_sums <= 0.0].index.tolist()
        raise ValueError(f"Cannot normalize zero-exposure weight rows: {bad_dates[:5]}")
    prepared = prepared.div(row_sums, axis=0)
    for date_value in prepared.index:
        residual = 1.0 - float(prepared.loc[date_value].sum())
        if abs(residual) <= 1e-12:
            continue
        target_ticker = prepared.loc[date_value].idxmax()
        prepared.loc[date_value, target_ticker] += residual
    return validate_weights_frame(prepared, dataset_name=spec_obj, repo_root=repo_root)


In [5]:
def _run_eamon_final_regime_backtest(dataset_name: str, model_obj: xgb.XGBClassifier) -> dict[str, object]:
    spec_obj = get_dataset_spec(dataset_name, repo_root=repo_root)
    benchmark = spec_obj.benchmark_ticker.upper()
    tradable = [ticker.upper() for ticker in spec_obj.tickers if ticker.upper() != benchmark]
    if not tradable:
        raise ValueError(f"{dataset_name} has no tradable tickers after excluding the benchmark.")

    loaded_prices = load_prices(spec_obj, repo_root=repo_root)
    loaded_prices = loaded_prices.copy()
    loaded_prices["date"] = pd.to_datetime(loaded_prices["date"], utc=True).dt.tz_localize(None)
    loaded_prices["ticker"] = loaded_prices["ticker"].astype(str).str.upper()
    price_wide = loaded_prices.pivot(index="date", columns="ticker", values="adj_close").sort_index()
    returns_wide = price_wide.reindex(columns=tradable).pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)

    latest_prices = price_wide.reindex(columns=tradable).ffill().iloc[-1]
    dataset_output_dir = EAMON_FINAL_REGIME_BACKTEST_OUTPUT_DIR / dataset_name
    dataset_output_dir.mkdir(parents=True, exist_ok=True)
    iv_snapshot = _eamon_final_fetch_current_iv_snapshot(
        tradable,
        latest_prices,
        dataset_output_dir / "current_iv_snapshot.csv",
        refresh=EAMON_FINAL_REFRESH_IV,
    )
    feature_prices = _eamon_final_prices_with_spy_context(loaded_prices, spec_obj)
    model_features = _eamon_final_build_model_features(feature_prices, tradable, iv_snapshot)

    factors, rf = _eamon_final_load_ff5_for_prices(loaded_prices)
    common_factor_dates = returns_wide.index.intersection(factors.index)
    test_dates = pd.DatetimeIndex(price_wide.loc[pd.Timestamp(spec_obj.test_start):pd.Timestamp(spec_obj.test_end)].index.unique()).sort_values()
    if test_dates.empty:
        raise ValueError(f"No test trading dates found for {dataset_name}: {spec_obj.test_start} to {spec_obj.test_end}")
    rebalance_dates = test_dates[::REBALANCE_EVERY_DAYS]

    feature_by_date = {pd.Timestamp(date): frame.copy() for date, frame in model_features.groupby("date", sort=True)}
    available_feature_dates = pd.DatetimeIndex(sorted(feature_by_date.keys()))
    if available_feature_dates.empty:
        raise ValueError(f"No model feature snapshots were built for {dataset_name}.")

    minvar_rows = []
    tilted_rows = []
    xgb_rows = []
    delta_rows = []
    score_rows = []
    diagnostics_rows = []

    print(f"{dataset_name}: {len(test_dates)} test trading dates, {len(rebalance_dates)} rebalance dates")
    for idx, rebalance_date in enumerate(rebalance_dates, start=1):
        fitted_tickers, sigma, beta_df, diagnostics = _eamon_final_estimate_ew_ff5_covariance(
            rebalance_date,
            returns_wide,
            tradable,
            factors,
            rf,
            common_factor_dates,
        )
        w_minvar = _eamon_final_solve_min_variance_weights(fitted_tickers, sigma)

        feature_pos = available_feature_dates.searchsorted(pd.Timestamp(rebalance_date), side="left") - 1
        if feature_pos < 0:
            raise ValueError(f"No feature snapshot before {rebalance_date} for {dataset_name}")
        signal_date = available_feature_dates[feature_pos]
        snapshot = feature_by_date[signal_date]
        snapshot = snapshot.loc[snapshot["ticker"].isin(fitted_tickers)].copy()
        if snapshot.empty:
            raise ValueError(f"No fitted tickers have features on {signal_date} for {dataset_name}")

        proba = model_obj.predict_proba(snapshot[TILT_FEATURE_NAMES].astype(float))
        if proba.shape[1] < 5:
            raise ValueError(f"Expected five XGBoost class probabilities, received shape {proba.shape}")
        top_q_prob = pd.Series(proba[:, 4], index=snapshot["ticker"], name="top_quintile_probability")
        signal = top_q_prob.reindex(fitted_tickers).fillna(top_q_prob.mean())
        signal = signal - signal.mean()
        if float(signal.abs().sum()) > 0.0:
            delta = signal / signal.abs().sum() * TILT_ALPHA
        else:
            delta = signal * 0.0

        raw_tilted = w_minvar.add(delta, fill_value=0.0)
        w_tilted = _eamon_final_project_long_only_capped_weights(raw_tilted)
        positive_scores = top_q_prob.reindex(fitted_tickers).clip(lower=0.0).fillna(0.0)
        if float(positive_scores.sum()) > 0.0:
            w_xgb = positive_scores / positive_scores.sum()
        else:
            w_xgb = pd.Series(1.0 / len(fitted_tickers), index=fitted_tickers)

        minvar_row = _eamon_final_expand_weight_series(w_minvar, tradable)
        tilted_row = _eamon_final_expand_weight_series(w_tilted, tradable)
        xgb_row = _eamon_final_expand_weight_series(w_xgb, tradable)
        delta_row = _eamon_final_expand_weight_series(w_tilted - w_minvar, tradable)
        for row in [minvar_row, tilted_row, xgb_row, delta_row]:
            row.name = pd.Timestamp(rebalance_date)
        minvar_rows.append(minvar_row)
        tilted_rows.append(tilted_row)
        xgb_rows.append(xgb_row)
        delta_rows.append(delta_row)
        diagnostics_rows.append({**diagnostics, "signal_date": pd.Timestamp(signal_date)})

        score_frame = pd.DataFrame(
            {
                "date": pd.Timestamp(rebalance_date),
                "signal_date": pd.Timestamp(signal_date),
                "ticker": snapshot["ticker"].to_numpy(),
                "top_quintile_probability": top_q_prob.reindex(snapshot["ticker"]).to_numpy(),
            }
        )
        for class_idx in range(proba.shape[1]):
            score_frame[f"class_{class_idx}_probability"] = proba[:, class_idx]
        score_rows.append(score_frame)

        if idx % 10 == 0 or idx == len(rebalance_dates):
            print(f"Processed {idx}/{len(rebalance_dates)} rebalances through {pd.Timestamp(rebalance_date).date()}")

    minvar_weights = pd.DataFrame(minvar_rows)
    tilted_weights = pd.DataFrame(tilted_rows)
    xgb_weights = pd.DataFrame(xgb_rows)
    tilt_deltas = pd.DataFrame(delta_rows)
    risk_diagnostics = pd.DataFrame(diagnostics_rows)
    xgb_scores = pd.concat(score_rows, ignore_index=True)
    for frame in [minvar_weights, tilted_weights, xgb_weights, tilt_deltas]:
        frame.index.name = "date"

    combined_abs_exposure = minvar_weights.abs().add(tilted_weights.abs(), fill_value=0.0).add(xgb_weights.abs(), fill_value=0.0)
    active_columns = combined_abs_exposure.columns[(combined_abs_exposure.sum(axis=0) > 0.0)]
    minvar_weights = validate_weights_frame(minvar_weights.loc[:, active_columns], dataset_name=spec_obj, repo_root=repo_root)
    tilted_weights = validate_weights_frame(tilted_weights.loc[:, active_columns], dataset_name=spec_obj, repo_root=repo_root)
    xgb_weights = validate_weights_frame(xgb_weights.loc[:, active_columns], dataset_name=spec_obj, repo_root=repo_root)
    tilt_deltas = tilt_deltas.loc[:, active_columns]

    minvar_weights_for_backtest = _eamon_final_prepare_backtest_weight_frame(minvar_weights, tradable, spec_obj)
    tilted_weights_for_backtest = _eamon_final_prepare_backtest_weight_frame(tilted_weights, tradable, spec_obj)
    portfolio_minvar = PortfolioWeights(
        weights=minvar_weights_for_backtest,
        dataset_name=spec_obj.identifier,
        strategy_name=f"{MINVAR_MODEL_NAME}_{dataset_name}",
        metadata={
            "type": "rolling_minvar",
            "factor_model": "ew_ff5",
            "prediction_horizon_days": PREDICTION_HORIZON_DAYS,
            "rebalance_every_days": REBALANCE_EVERY_DAYS,
            "ff_window": FF_WINDOW,
            "ew_halflife": EW_HALFLIFE,
        },
    )
    portfolio_tilted = PortfolioWeights(
        weights=tilted_weights_for_backtest,
        dataset_name=spec_obj.identifier,
        strategy_name=f"{MODEL_NAME}_{dataset_name}",
        metadata={
            "type": "rolling_minvar_xgb_tilt",
            "factor_model": "ew_ff5",
            "prediction_horizon_days": PREDICTION_HORIZON_DAYS,
            "rebalance_every_days": REBALANCE_EVERY_DAYS,
            "ff_window": FF_WINDOW,
            "ew_halflife": EW_HALFLIFE,
            "tilt_alpha": TILT_ALPHA,
            "iv_source": "current_yfinance_option_chain_static_by_ticker",
            "source_mlflow_run_id": eamon_final_mlflow_model_info["run_id"],
            "source_mlflow_artifact_path": eamon_final_mlflow_model_info["artifact_path"],
        },
    )

    result_minvar = backtest_weights(spec_obj, portfolio_minvar, benchmark=spec_obj.benchmark_ticker, repo_root=repo_root)
    result_tilted = backtest_weights(spec_obj, portfolio_tilted, benchmark=spec_obj.benchmark_ticker, repo_root=repo_root)
    artifact_paths_minvar = write_backtest_artifacts(result_minvar, dataset_output_dir / "minvar")
    artifact_paths_tilted = write_backtest_artifacts(result_tilted, dataset_output_dir / "tilted")

    minvar_weights.to_parquet(dataset_output_dir / "minvar_weights_raw.parquet")
    tilted_weights.to_parquet(dataset_output_dir / "tilted_weights_raw.parquet")
    xgb_weights.to_parquet(dataset_output_dir / "xgb_probability_weights.parquet")
    tilt_deltas.to_parquet(dataset_output_dir / "tilt_deltas.parquet")
    risk_diagnostics.to_parquet(dataset_output_dir / "risk_diagnostics.parquet", index=False)
    xgb_scores.to_parquet(dataset_output_dir / "xgb_probability_scores.parquet", index=False)
    pd.DataFrame([result_tilted.metrics]).to_parquet(dataset_output_dir / "tilted_metrics.parquet", index=False)
    pd.DataFrame([result_minvar.metrics]).to_parquet(dataset_output_dir / "minvar_metrics.parquet", index=False)

    return {
        "dataset_name": dataset_name,
        "spec": spec_obj,
        "portfolio": portfolio_tilted,
        "portfolio_minvar": portfolio_minvar,
        "result": result_tilted,
        "result_minvar": result_minvar,
        "risk_diagnostics": risk_diagnostics,
        "xgb_scores": xgb_scores,
        "output_dir": dataset_output_dir,
        "artifact_paths_tilted": artifact_paths_tilted,
        "artifact_paths_minvar": artifact_paths_minvar,
        "rebalance_count": len(rebalance_dates),
        "active_ticker_count": len(tilted_weights_for_backtest.columns),
        "iv_missing_count": int(iv_snapshot["current_iv_missing"].sum()),
    }


eamon_final_xgb_model, eamon_final_mlflow_model_info = _eamon_final_load_xgb_model_from_mlflow()
eamon_final_regime_backtest_runs = {}
for dataset_name in EAMON_FINAL_REGIME_BACKTEST_DATASETS:
    print(f"Running Eamon final four-regime backtest: {dataset_name}", flush=True)
    eamon_final_regime_backtest_runs[dataset_name] = _run_eamon_final_regime_backtest(dataset_name, eamon_final_xgb_model)

print("Completed Eamon final regime backtests:", list(eamon_final_regime_backtest_runs))


Loaded Eamon XGBoost model from MLflow:
{
  "run_id": "638ef484f9c2425db1f54fa001d2b806",
  "artifact_path": "model_submission/artifacts/eamon_xgboost_final.json",
  "local_model_path": "/var/folders/3y/dhkqtqns7p35w8t4_svmjz780000gn/T/tmpwahoh80y/eamon_xgboost_final.json"
}
Running Eamon final four-regime backtest: regime_modern_tech_gain_2022_2026
Fetched IV for 25/40 tickers
Fetched IV for 40/40 tickers
regime_modern_tech_gain_2022_2026: 1100 test trading dates, 110 rebalance dates
Processed 10/110 rebalances through 2022-05-12
Processed 20/110 rebalances through 2022-10-05
Processed 30/110 rebalances through 2023-03-01
Processed 40/110 rebalances through 2023-07-25
Processed 50/110 rebalances through 2023-12-14
Processed 60/110 rebalances through 2024-05-09
Processed 70/110 rebalances through 2024-10-02
Processed 80/110 rebalances through 2025-02-27
Processed 90/110 rebalances through 2025-07-23
Processed 100/110 rebalances through 2025-12-12
Processed 110/110 rebalances through 20

In [6]:
eamon_final_summary_rows = []
for dataset_name, payload in eamon_final_regime_backtest_runs.items():
    spec_obj = payload["spec"]
    result_obj = payload["result"]
    minvar_result_obj = payload["result_minvar"]
    metrics_obj = dict(result_obj.metrics)
    minvar_metrics_obj = dict(minvar_result_obj.metrics)
    total_return = float(metrics_obj.get("total_return", np.nan))
    ending_bankroll = EAMON_FINAL_REGIME_BACKTEST_ALLOCATION * (1.0 + total_return)
    eamon_final_summary_rows.append(
        {
            "dataset_name": dataset_name,
            "benchmark_ticker": spec_obj.benchmark_ticker,
            "test_start": spec_obj.test_start,
            "test_end": spec_obj.test_end,
            "allocation": EAMON_FINAL_REGIME_BACKTEST_ALLOCATION,
            "ending_bankroll": ending_bankroll,
            "profit_loss": ending_bankroll - EAMON_FINAL_REGIME_BACKTEST_ALLOCATION,
            "total_return": total_return,
            "annual_return": metrics_obj.get("annual_return", np.nan),
            "annual_volatility": metrics_obj.get("annual_volatility", np.nan),
            "sharpe": metrics_obj.get("sharpe", np.nan),
            "sortino": metrics_obj.get("sortino", np.nan),
            "max_drawdown": metrics_obj.get("max_drawdown", np.nan),
            "average_turnover": metrics_obj.get("average_turnover", np.nan),
            "benchmark_total_return": metrics_obj.get("benchmark_total_return", np.nan),
            "excess_return_vs_benchmark": metrics_obj.get("excess_return_vs_benchmark", np.nan),
            "minvar_total_return": minvar_metrics_obj.get("total_return", np.nan),
            "tilt_excess_vs_minvar": total_return - float(minvar_metrics_obj.get("total_return", np.nan)),
            "rebalance_count": payload["rebalance_count"],
            "active_ticker_count": payload["active_ticker_count"],
            "iv_missing_count": payload["iv_missing_count"],
            "avg_active_names": float((result_obj.weights > 1e-8).sum(axis=1).mean()),
            "avg_max_weight": float(result_obj.weights.max(axis=1).mean()),
        }
    )

eamon_final_regime_backtest_summary = pd.DataFrame(eamon_final_summary_rows)
eamon_final_combined_ending_bankroll = float(eamon_final_regime_backtest_summary["ending_bankroll"].sum())
eamon_final_combined_profit_loss = eamon_final_combined_ending_bankroll - EAMON_FINAL_REGIME_BACKTEST_TOTAL_BANKROLL
eamon_final_combined_total_return = eamon_final_combined_profit_loss / EAMON_FINAL_REGIME_BACKTEST_TOTAL_BANKROLL
eamon_final_combined_row = pd.DataFrame(
    [
        {
            "dataset_name": "COMBINED_4_REGIME_BANKROLL",
            "benchmark_ticker": "mixed",
            "test_start": min(eamon_final_regime_backtest_summary["test_start"]),
            "test_end": max(eamon_final_regime_backtest_summary["test_end"]),
            "allocation": EAMON_FINAL_REGIME_BACKTEST_TOTAL_BANKROLL,
            "ending_bankroll": eamon_final_combined_ending_bankroll,
            "profit_loss": eamon_final_combined_profit_loss,
            "total_return": eamon_final_combined_total_return,
            "annual_return": np.nan,
            "annual_volatility": np.nan,
            "sharpe": np.nan,
            "sortino": np.nan,
            "max_drawdown": np.nan,
            "average_turnover": np.nan,
            "benchmark_total_return": np.nan,
            "excess_return_vs_benchmark": np.nan,
            "minvar_total_return": np.nan,
            "tilt_excess_vs_minvar": np.nan,
            "rebalance_count": int(eamon_final_regime_backtest_summary["rebalance_count"].sum()),
            "active_ticker_count": np.nan,
            "iv_missing_count": int(eamon_final_regime_backtest_summary["iv_missing_count"].sum()),
            "avg_active_names": np.nan,
            "avg_max_weight": np.nan,
        }
    ]
)
eamon_final_regime_backtest_summary_with_total = pd.concat(
    [eamon_final_regime_backtest_summary, eamon_final_combined_row],
    ignore_index=True,
)

eamon_final_summary_path = EAMON_FINAL_REGIME_BACKTEST_OUTPUT_DIR / "four_regime_bankroll_summary.parquet"
eamon_final_csv_summary_path = EAMON_FINAL_REGIME_BACKTEST_OUTPUT_DIR / "four_regime_bankroll_summary.csv"
eamon_final_regime_backtest_summary_with_total.to_parquet(eamon_final_summary_path, index=False)
eamon_final_regime_backtest_summary_with_total.to_csv(eamon_final_csv_summary_path, index=False)

print(f"Starting bankroll: ${EAMON_FINAL_REGIME_BACKTEST_TOTAL_BANKROLL:,.0f}")
print(f"Ending bankroll:   ${eamon_final_combined_ending_bankroll:,.0f}")
print(f"Total P/L:         ${eamon_final_combined_profit_loss:,.0f}")
print(f"Combined return:   {eamon_final_combined_total_return:.2%}")
display(eamon_final_regime_backtest_summary_with_total)


Starting bankroll: $4,800,000
Ending bankroll:   $9,425,017
Total P/L:         $4,625,017
Combined return:   96.35%


,dataset_name,benchmark_ticker,test_start,test_end,allocation,ending_bankroll,profit_loss,total_return,annual_return,annual_volatility,...,average_turnover,benchmark_total_return,excess_return_vs_benchmark,minvar_total_return,tilt_excess_vs_minvar,rebalance_count,active_ticker_count,iv_missing_count,avg_active_names,avg_max_weight
0,regime_modern_tech_gain_2022_2026,XLK,2022-01-03,2026-05-21,1200000.0,2.251486e+06,1.051486e+06,0.876239,0.154584,0.233314,...,0.097401,1.100637,-0.224398,0.642419,0.233820,110,40.0,0,21.809091,0.146036
1,regime_financial_crisis_loss_2005_2010,XLF,2005-01-03,2010-12-31,1200000.0,1.042711e+06,-1.572893e+05,-0.131074,-0.023181,0.333822,...,0.102409,-0.392930,0.261856,-0.232010,0.100935,152,33.0,1,18.671053,0.145521
2,regime_nineties_volatility_1995_1999,MDY,1995-05-05,1999-12-31,1200000.0,3.939660e+06,2.739660e+06,2.283050,0.290799,0.151327,...,0.120477,1.407843,0.875207,1.833218,0.449832,118,41.0,0,27.432203,0.127538
3,regime_oil_pre2014_energy_2010_2013,XOP,2010-01-04,2013-12-31,1200000.0,2.191160e+06,9.911601e+05,0.825967,0.162928,0.213241,...,0.088278,0.645439,0.180527,0.766182,0.059785,101,22.0,0,14.891089,0.145478
4,COMBINED_4_REGIME_BANKROLL,mixed,1995-05-05,2026-05-21,4800000.0,9.425017e+06,4.625017e+06,0.963545,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,481,NaN,1,NaN,NaN


In [7]:
if EAMON_FINAL_FOUR_REGIME_LOG_TO_MLFLOW:
    init_mlflow(repo_root=repo_root)
    for dataset_name, payload in eamon_final_regime_backtest_runs.items():
        spec_obj = payload["spec"]
        result_obj = payload["result"]
        portfolio_obj = payload["portfolio"]
        bankroll_row = eamon_final_regime_backtest_summary.loc[
            eamon_final_regime_backtest_summary["dataset_name"] == dataset_name
        ].iloc[0]
        with start_run(
            run_name=f"{MODEL_NAME}_four_regime_{dataset_name}",
            dataset_name=spec_obj,
            tags={
                "workflow": "eamon_final_four_regime_proxy_backtest",
                "model_family": "xgboost",
                "strategy_type": "rolling_minvar_xgb_tilt",
                "factor_model": "ew_ff5",
                "source_mlflow_run_id": eamon_final_mlflow_model_info["run_id"],
                "source_mlflow_artifact_path": eamon_final_mlflow_model_info["artifact_path"],
                "prediction_horizon_days": str(PREDICTION_HORIZON_DAYS),
                "rebalance_every_days": str(REBALANCE_EVERY_DAYS),
                "bankroll_allocation": str(EAMON_FINAL_REGIME_BACKTEST_ALLOCATION),
            },
            repo_root=repo_root,
        ):
            mlflow.log_params(
                {
                    "model_name": MODEL_NAME,
                    "source_mlflow_run_id": eamon_final_mlflow_model_info["run_id"],
                    "source_mlflow_artifact_path": eamon_final_mlflow_model_info["artifact_path"],
                    "dataset_name": dataset_name,
                    "benchmark_ticker": spec_obj.benchmark_ticker,
                    "test_start": str(spec_obj.test_start),
                    "test_end": str(spec_obj.test_end),
                    "prediction_horizon_days": PREDICTION_HORIZON_DAYS,
                    "rebalance_every_days": REBALANCE_EVERY_DAYS,
                    "ff_window": FF_WINDOW,
                    "ew_halflife": EW_HALFLIFE,
                    "tilt_alpha": TILT_ALPHA,
                    "min_weight": MIN_WEIGHT,
                    "max_weight": MAX_WEIGHT,
                    "feature_count": len(TILT_FEATURE_NAMES),
                    "iv_source": "current_yfinance_option_chain_static_by_ticker",
                    "iv_missing_count": payload["iv_missing_count"],
                    "rebalance_count": payload["rebalance_count"],
                    "active_ticker_count": payload["active_ticker_count"],
                    "bankroll_allocation": EAMON_FINAL_REGIME_BACKTEST_ALLOCATION,
                }
            )
            mlflow.log_metrics(
                {
                    "bankroll_allocation": float(bankroll_row["allocation"]),
                    "ending_bankroll": float(bankroll_row["ending_bankroll"]),
                    "profit_loss": float(bankroll_row["profit_loss"]),
                    "avg_active_names": float(bankroll_row["avg_active_names"]),
                    "avg_max_weight": float(bankroll_row["avg_max_weight"]),
                    "minvar_total_return": float(bankroll_row["minvar_total_return"]),
                    "tilt_excess_vs_minvar": float(bankroll_row["tilt_excess_vs_minvar"]),
                }
            )
            log_portfolio(portfolio_obj)
            log_backtest(result_obj)
            for artifact_name in [
                "risk_diagnostics.parquet",
                "xgb_probability_scores.parquet",
                "xgb_probability_weights.parquet",
                "tilt_deltas.parquet",
                "tilted_metrics.parquet",
                "minvar_metrics.parquet",
            ]:
                artifact_path = payload["output_dir"] / artifact_name
                if artifact_path.exists():
                    mlflow.log_artifact(str(artifact_path))
    with start_run(
        run_name=f"{MODEL_NAME}_four_regime_combined_bankroll",
        dataset_name=EAMON_FINAL_REGIME_BACKTEST_DATASETS[0],
        tags={
            "workflow": "eamon_final_four_regime_proxy_backtest_summary",
            "combined_backtest": "true",
            "source_mlflow_run_id": eamon_final_mlflow_model_info["run_id"],
            "total_bankroll": str(EAMON_FINAL_REGIME_BACKTEST_TOTAL_BANKROLL),
        },
        repo_root=repo_root,
    ):
        mlflow.log_metrics(
            {
                "total_bankroll": EAMON_FINAL_REGIME_BACKTEST_TOTAL_BANKROLL,
                "combined_ending_bankroll": eamon_final_combined_ending_bankroll,
                "combined_profit_loss": eamon_final_combined_profit_loss,
                "combined_total_return": eamon_final_combined_total_return,
            }
        )
        mlflow.log_artifact(str(eamon_final_summary_path))
        mlflow.log_artifact(str(eamon_final_csv_summary_path))
    print("Eamon final four-regime MLflow logging complete.")
else:
    print("Eamon final four-regime MLflow logging skipped because EAMON_FINAL_FOUR_REGIME_LOG_TO_MLFLOW=0.")


🏃 View run eamon_xgboost_final_four_regime_regime_modern_tech_gain_2022_2026 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/10/runs/0c15941306114a10a99346d0731c1e9c
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/10
🏃 View run eamon_xgboost_final_four_regime_regime_financial_crisis_loss_2005_2010 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/11/runs/2faa2c853d5d4fc29f9da769e74510f7
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/11
🏃 View run eamon_xgboost_final_four_regime_regime_nineties_volatility_1995_1999 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/12/runs/e5b86c0fd20945e58f19b866224c33e4
🧪 View experiment at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/12
🏃 View run eamon_xgboost_final_four_regime_regime_oil_pre2014_energy_2010_2013 at: https://adams-macbook-pro.tail5ddc35.ts.net/#/experiments/13/runs/c8180eb151d5472c8ab533df8a17fd3c
🧪 View experiment at: